In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  
N = 8000


def flag(p):
    # Return 0/1 flags with probability p (single value or per-row array)
    return (rng.random(N) < p).astype(int)


# Demographics
df = pd.DataFrame({
    "Customer_ID": [f"AB-{i:05d}" for i in range(10001, 10001 + N)],
    "Age": rng.integers(25, 66, N),
    "Income_Bracket": rng.choice(["Low", "Medium", "High"], N, p=[0.2, 0.6, 0.2]),
})

# Independent triggers (column: probability of 1)
triggers = {
    "Marital_Status_Changed": 0.15,
    "Has_Dependent_Children": 0.60,
    "Recent_Mortgage_Indicator": 0.10,
    "Divorce_Indicator": 0.08,
    "Business_Owner_Indicator": 0.12,
    "Other_Policy_Renewal_30D": 0.20,
    "Significant_Income_Increase": 0.15,
}
for col, p in triggers.items():
    df[col] = flag(p)

# Milestone birthday (turning 30, 35, 40...) within 6 months, 50% chance if eligible
df["Milestone_Birthday_Within_6M"] = flag(0.5) * ((df["Age"] + 1) % 5 == 0)

# Marketing engagement (some depend on the triggers above)
df["Website_Visits_30D"] = (
    rng.poisson(2, N)
    + df["Marital_Status_Changed"] * 3
    + df["Recent_Mortgage_Indicator"] * 4
    + df["Business_Owner_Indicator"] * 2
)
df["Premium_Calculator_Used"] = flag(
    0.15 + 0.40 * df["Recent_Mortgage_Indicator"] + 0.20 * df["Significant_Income_Increase"]
)
df["Insurance_Guide_Downloads"] = (
    rng.choice([0, 1, 2], N, p=[0.7, 0.2, 0.1])
    + df["Marital_Status_Changed"]
    + df["Divorce_Indicator"]
)
df["Email_Open_Rate_Pct"] = np.clip(
    rng.uniform(5, 85, N)
    + 15 * df["Has_Dependent_Children"]
    + 10 * df["Other_Policy_Renewal_30D"],
    0, 100,
).round(1)

# Business columns
df["Lead_Source"] = rng.choice(
    ["Website", "Referral", "Paid Social", "Partner Bank", "Email Campaign"],
    N, p=[0.30, 0.15, 0.25, 0.10, 0.20],
)
df["Product_Interest"] = rng.choice(
    ["Term Life", "Whole Life", "Critical Illness", "Family Protection"],
    N, p=[0.40, 0.15, 0.20, 0.25],
)
df["Assigned_Agent"] = rng.choice([f"Agent_{i:02d}" for i in range(1, 11)], N)
df["Lead_Created_Date"] = pd.Series(
    rng.choice(pd.date_range("2025-10-01", "2026-09-30"), N)
).dt.date

# Annual premium: driven by income and product, with realistic noise
income_base = df["Income_Bracket"].map({"Low": 600, "Medium": 1200, "High": 2400})
product_factor = df["Product_Interest"].map(
    {"Term Life": 1.0, "Whole Life": 1.8, "Critical Illness": 1.3, "Family Protection": 1.2}
)
df["Annual_Premium"] = (income_base * product_factor * rng.lognormal(0, 0.25, N)).round(-1).astype(int)

# Historical conversion (vectorised)
weights = {
    "Marital_Status_Changed": 0.25,
    "Recent_Mortgage_Indicator": 0.30,
    "Has_Dependent_Children": 0.15,
    "Divorce_Indicator": 0.20,
    "Business_Owner_Indicator": 0.15,
    "Other_Policy_Renewal_30D": 0.15,
    "Premium_Calculator_Used": 0.25,
    "Significant_Income_Increase": 0.15,
    "Milestone_Birthday_Within_6M": 0.10,
}
score = (
    0.05
    + 0.10 * df["Age"].between(30, 45)  # peak buying age
    + sum(df[col] * w for col, w in weights.items())
    + np.minimum(df["Website_Visits_30D"] * 0.03, 0.15)
    + np.minimum(df["Insurance_Guide_Downloads"] * 0.10, 0.20)
)

# Channel effect, so the "conversion by source" chart is not flat
source_boost = df["Lead_Source"].map(
    {"Referral": 0.06, "Partner Bank": 0.03, "Website": 0.0, "Email Campaign": 0.0, "Paid Social": -0.03}
)
prob = (score * 0.35 + source_boost).clip(0.02, 0.95)
df["Converted_Historically"] = flag(prob)

df.to_csv("insurance_synthetic_lead_data_broad.csv", index=False)
print(f"Saved {N} rows. Overall conversion: {df['Converted_Historically'].mean():.1%}")

Saved 8000 rows. Overall conversion: 18.8%


In [2]:
import duckdb

SCORING = """
SELECT *,
    (5
     + CASE WHEN Age BETWEEN 30 AND 45 THEN 10 ELSE 0 END
     + Marital_Status_Changed * 25
     + Recent_Mortgage_Indicator * 30
     + Has_Dependent_Children * 15
     + Divorce_Indicator * 20
     + Business_Owner_Indicator * 15
     + Other_Policy_Renewal_30D * 15
     + Premium_Calculator_Used * 25
     + Significant_Income_Increase * 15
     + Milestone_Birthday_Within_6M * 10
     + LEAST(Website_Visits_30D * 3, 15)
     + LEAST(Insurance_Guide_Downloads * 10, 20)
    ) AS Lead_Score
FROM 'insurance_synthetic_lead_data_broad.csv'
"""

# Check the score distribution and define cutoffs
duckdb.sql(f"""
    SELECT MIN(Lead_Score) AS min, MAX(Lead_Score) AS max,
           quantile_cont(Lead_Score, [0.40, 0.80, 0.95]) AS p40_p80_p95
    FROM ({SCORING})
""").show()

# Create final table. Cutoffs @ p40, p80, p95
final_query = f"""
SELECT
    Customer_ID, 
    Age,
    CASE WHEN Age < 35 THEN '25-34' 
        WHEN Age < 45 THEN '35-44'
         WHEN Age < 55 THEN '45-54' 
         ELSE '55-65' END AS Age_Band,
    Income_Bracket, 
    Lead_Source, 
    Product_Interest, 
    Assigned_Agent,
    Lead_Created_Date, 
    Annual_Premium, 
    Lead_Score,
    CASE WHEN Lead_Score > 109 THEN 'Platinum (Elite)'
        WHEN Lead_Score > 78  THEN 'Gold (High)'
        WHEN Lead_Score > 42  THEN 'Silver (Medium)'
        ELSE 'Bronze (Low)' END AS Lead_Tier,
    Converted_Historically
FROM ({SCORING})
"""

duckdb.sql(f"COPY ({final_query}) TO 'insurance_scored_leads_broad.csv' (HEADER, DELIMITER ',')")

# Sanity check: conversion should rise clearly from Bronze to Platinum
duckdb.sql(f"""
    SELECT Lead_Tier, COUNT(*) AS leads,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_leads,
           ROUND(100.0 * AVG(Converted_Historically), 1) AS conv_pct
    FROM ({final_query}) GROUP BY Lead_Tier ORDER BY conv_pct DESC

""").show()

┌───────┬───────┬─────────────────────┐
│  min  │  max  │     p40_p80_p95     │
│ int64 │ int64 │      double[]       │
├───────┼───────┼─────────────────────┤
│     5 │   195 │ [42.0, 78.0, 109.0] │
└───────┴───────┴─────────────────────┘

┌──────────────────┬───────┬──────────────┬──────────┐
│    Lead_Tier     │ leads │ pct_of_leads │ conv_pct │
│     varchar      │ int64 │    double    │  double  │
├──────────────────┼───────┼──────────────┼──────────┤
│ Platinum (Elite) │   396 │          5.0 │     41.4 │
│ Gold (High)      │  1188 │         14.9 │     31.6 │
│ Silver (Medium)  │  3134 │         39.2 │     19.4 │
│ Bronze (Low)     │  3282 │         41.0 │     10.9 │
└──────────────────┴───────┴──────────────┴──────────┘

